# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">6주차 · FM 과 MLP 로 상품 정보를 넣어 보고, 위험도 상한으로 추천을 거르기</mark>

지금까지 모델은 **누가 무엇을 담았는지(거래 기록)만** 봤습니다.

종목이 레버리지인지, 위험도가 몇인지는 **한 번도 보지 않았습니다.**

오늘은 종목마다 적혀 있는 정보인 **상품 정보(item feature)** 를 씁니다. 쓰는 방법은 두 가지입니다.

1. **모델에 넣는다** — 상품 정보까지 입력으로 받는 모델 두 개를 직접 학습해 본다
   - **FM** (factorization machine) — 4주차 행렬분해를 넓힌 모델
   - **MLP** (다층 신경망) — 딥러닝의 기본형
2. **위험도 상한을 건다** — 위험도가 투자 성향에 맞지 않는 종목을 추천 목록에서 뺀다

**오늘의 결론을 미리 말하면** — 우리처럼 **데이터가 작을 때는 단순한 모델이 더 낫습니다.** 왜 그런지를 숫자로 확인합니다.

---

### 오늘의 구성 — 약 50분 (PPT + 노트북)

| 시간 | 어디서 | 하는 일 |
|---|---|---|
| 여는 순서 4분 | PPT | 지난 다섯 주 · 안정형 U0001 에게 간 위험도 5 |
| 개념 19분 | PPT | 투자성향 · 적합성원칙 · 상품 정보 · 원핫 인코딩 · **FM 의 짝 구조** · MLP 와 층 · **과적합** · 후처리 필터링 |
| 실습 22분 | **이 노트북** | 맨 위부터 실행 → 6.5 · 6.6 점수 읽기 → 6.7 비교 표 → 🗣 함께 보기 → 6.8 · 6.9 위험도 상한 |
| 마무리 5분 | PPT | 용어 정리 · 점수판 · 혼자 해 보기 · 다음 주 |

### 이 노트북 읽는 법 — 🎯 와 📖

- **🎯 수업** — 수업 시간에 다루는 절입니다
  - 셀이 있으면 실행하고 **결과 숫자를 함께 읽습니다**
- **📖 혼자 읽기** — 수업에서는 건너뛰고 **복습할 때 읽는** 절입니다
  - 6.2 · 6.3 · 6.10 이 여기에 속합니다

⏱ **실습을 시작하면 먼저 메뉴의 `런타임 → 모두 실행` 을 눌러 두세요.** 전체 실행에 30초쯤 걸립니다. 그동안 위에서부터 결과를 읽어 내려가면 됩니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/06_item_features.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것**

| 무엇 | 쓰임 |
|---|---|
| `data/items.csv` | 종목 100개 — 이름 · 섹터 · 자산군 · **위험도** · 테마 |
| `data/users.csv` | 투자자 300명 — **투자 성향**(안정형 · 성장형 · 공격형) |
| `data/interactions.csv` | 거래 기록 8,086건 |
| `recsys.py` | 데이터 읽기 · 시간순 분할 · 채점 · 사용자-아이템 행렬 · **오늘 새로: 학습 데이터 만들기 · FM** |

⚠ **지난주 노트북을 먼저 돌리지 않아도 됩니다.** 이 노트북 하나로 끝납니다.

⚠ **따로 설치할 것이 없습니다.** 오늘 쓰는 도구(numpy · pandas · scikit-learn)는 Colab 에 처음부터 들어 있습니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 일곱 개</mark>

**1\~2번은 재료, 3\~4번은 모델, 5번은 학습 결과, 6\~7번은 금융 규칙입니다.**

새 말이 많습니다. 여기서 다 외우지 않아도 됩니다. **노트북에서 그 말이 처음 쓰이는 절에서 예와 함께 다시 설명합니다.** 괄호 안이 그 절입니다.

#### 재료

1. **상품 정보 (item feature)** — 거래 기록이 아니라 **종목 자체에 대한 정보** (6.3)
   - 오늘 쓰는 것은 `items.csv` 의 열(column) 네 개 — 섹터 · 자산군 · 위험도 · 테마
   - 네 가지가 각각 무엇인지, 어떤 값이 적혀 있는지는 바로 아래 표에 있습니다

| 열 이름 | 한글 | 뜻 | 값의 종류 | 예 |
|---|---|---|---|---|
| `sector` | **섹터** | 그 종목이 속한 **업종 · 분야** | 25가지 — 지수 14개 · 반도체 13개 · 금융 11개 · 레버리지 6개 · 2차전지 6개 등 | Apple → `기술` · 삼성생명 → `금융` · Taiwan Semiconductor → `반도체` · 크래프톤 → `게임` |
| `asset_class` | **자산군** | 그 종목이 **어떤 종류의 상품인가** | 2가지 — **주식** 52개 · **ETF** 48개 | Apple → `주식` · KODEX 200 → `ETF` |
| `risk_level` | **위험도** | 원금을 잃을 가능성의 **크기** | 5단계 — 1(가장 낮음) 4개 · 2는 18개 · 3은 30개 · 4는 28개 · 5(가장 높음) 20개 | KODEX 단기채권 → `1` · Procter & Gamble → `2` · 삼성생명 → `3` · 크래프톤 → `4` · KODEX 레버리지 → `5` |
| `theme` | **테마** | 그 종목의 **투자 성격 · 투자하는 이유** | 9가지 — 성장 40개 · 배당 18개 · 인덱스 10개 · 가치 9개 · 레버리지 8개 등 | Apple → `성장` · 삼성생명 → `배당` · KODEX 200 → `인덱스` · TIGER 2차전지테마 → `친환경` |

- **자산군에서 ETF 란** — 상장지수펀드(Exchange Traded Fund). 여러 종목을 묶어 하나의 상품으로 만들고, 그 상품을 **주식처럼 사고파는** 상품입니다
  - 예 — 「KODEX 레버리지」는 한 회사의 주식이 아니라 지수를 따라가는 ETF 입니다
- **위험도는 숫자가 클수록 위험합니다** — 5 가 가장 위험합니다
  - 실제 금융 상품의 위험등급은 1등급이 가장 위험해서 방향이 반대입니다(아래 7번에서 다시 설명합니다)
- **섹터와 테마의 차이** — 섹터는 「**무엇을 하는** 회사 · 상품인가」, 테마는 「**왜 사는** 상품인가」입니다
  - 같은 섹터라도 테마가 다를 수 있고, 「레버리지」처럼 두 열에 같은 말이 들어가기도 합니다

종목 다섯 개로 보면 이렇게 적혀 있습니다.

| 종목 | 섹터 | 자산군 | 위험도 | 테마 |
|---|---|---|---|---|
| KODEX 단기채권 | 채권 | ETF | **1** | 채권 |
| TIGER 미국배당다우존스 | 배당 | ETF | 2 | 배당 |
| 삼성생명 | 금융 | 주식 | 3 | 배당 |
| Apple | 기술 | 주식 | 3 | 성장 |
| KODEX 레버리지 | 레버리지 | ETF | **5** | 레버리지 |

- 이 네 가지가 종목마다 하나씩 정해져 있고, 거래 기록과는 **상관없이 종목 자체에 적혀 있는 정보**입니다
   - 2주차의 **피처(feature)** 와 같은 말입니다. 그때는 사람에 대한 피처(주력 섹터 · 평균 위험도)였고, 오늘은 종목에 대한 피처입니다

2. **원핫 인코딩 (one-hot encoding)** — 글자로 된 범주를 **「그 범주면 1, 아니면 0」인 열 여러 개**로 바꾸는 방법 (6.3)
   - 이유 — 글자(`레버리지`)로는 곱하거나 더할 수 없습니다
   - 범주 하나마다 열이 하나 생기고, 한 줄에서 **1 이 들어가는 열은 하나뿐**이라서 「원핫(one-hot)」이라고 부릅니다
   - 예 — 종목 세 개의 섹터를 원핫 인코딩하면 아래 ① → ② 처럼 바뀝니다

① **원래** — 섹터가 글자로 적혀 있습니다.

| 종목 | 섹터 |
|---|---|
| KODEX 레버리지 | 레버리지 |
| 삼성생명 | 금융 |
| Procter & Gamble | 소비재 |

② **원핫 인코딩 후** — 섹터 하나마다 열을 하나 만들고, 그 섹터면 1, 아니면 0 을 적습니다.

| 종목 | 레버리지 | 금융 | 소비재 |
|---|---|---|---|
| KODEX 레버리지 | **1** | 0 | 0 |
| 삼성생명 | 0 | **1** | 0 |
| Procter & Gamble | 0 | 0 | **1** |


#### 모델

3. **FM (factorization machine, 팩토리제이션 머신)** — 거래 기록에 **상품 정보까지 함께** 써서, 「이 투자자가 이 종목을 담을 가능성」을 점수로 매기는 모델 (6.4 \~ 6.5)
   - 4주차 행렬분해는 투자자 번호와 종목 번호만 입력으로 썼습니다
   - FM 은 여기에 「섹터 반도체」 「위험도 5」 같은 **상품 정보를 입력으로 더합니다**
   - 입력의 항목을 **둘씩 짝지어** 점수를 계산합니다. 그래서 「투자자 × 위험도 5」 같은 짝을 따로 볼 수 있습니다
   - 예 — 위험도 5 는 공격형 투자자에게는 담을 이유가 되고, 안정형 투자자에게는 담지 않을 이유가 됩니다. 항목 하나로는 이 차이를 나타낼 수 없고, 짝으로 볼 때 나타낼 수 있습니다

4. **MLP (multi-layer perceptron, 다층 신경망)** — 계산 단계(**층**)를 여러 개 이어서 담을 확률을 내는 모델 (6.6)
   - 층을 많이 쌓은 모델을 「깊다(deep)」고 하고, 이것이 **딥러닝(deep learning)** 이라는 이름의 뜻입니다
   - 층이 많을수록 **학습으로 정해야 하는 숫자**가 늘어납니다. 오늘은 1층 모델이 약 1만 4천 개, 2층 모델이 약 2만 9천 개입니다
   - FM 은 계산 방법을 사람이 정해 두고 약 3,800개의 숫자만 학습합니다. MLP 는 계산 단계만 정해 두고 훨씬 많은 숫자를 학습합니다
   - 그래서 복잡한 관계를 표현할 수 있지만, 그만큼 **더 많은 데이터가 필요합니다** (6.7 에서 숫자로 확인합니다)

#### 학습 결과

5. **과적합 (overfitting)** — 모델이 **학습 데이터에만 지나치게 맞춰져서**, 학습 데이터는 잘 맞히는데 **새 데이터는 못 맞히는** 상태 (6.7)
   - 모델이 복잡할수록(고칠 숫자가 많을수록), 데이터가 적을수록 잘 생깁니다
   - 예 — MLP 2층은 학습 데이터의 줄 가운데 **98%** 를 맞혔지만, 다음 달 기록은 네 모델 중 가장 못 맞혔습니다 (6.7)

#### 금융 규칙

6. **적합성원칙 (suitability rule)** — 금융 회사는 **고객의 투자 성향에 맞지 않는 상품을 권유하면 안 된다**는 원칙 (6.8)
   - 금융소비자보호법 제17조에 있습니다
   - **투자성향**은 고객이 가입할 때 설문으로 정합니다 — 실제는 5단계, 우리 데이터는 안정형 · 성장형 · 공격형 3단계
   - 실제 **상품 위험등급**은 6단계이고 **1등급이 가장 위험**합니다. 우리 데이터의 위험도(1\~5)는 5가 가장 위험해서 방향이 반대입니다

7. **위험도 상한 (risk cap)** — 투자 성향마다 정한 **추천해도 되는 위험도의 최댓값** (6.9)
   - 이 값을 넘는 종목은 추천 목록에서 뺍니다
   - 모델이 만든 목록에서 규칙으로 종목을 빼는 것을 **후처리 필터링(post-filtering)** 이라고 합니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 — 데이터 읽기와 시간순 분할</mark>

**여기서 하는 일** — 데이터를 읽고, 3주차부터 쓰던 **시간순 분할**로 학습 구간과 채점 구간을 나눕니다.

**채점 기준이 같아야** 오늘 숫자를 4주차 **0.3918** 과 비교할 수 있습니다.

In [2]:
# 이 셀에서 하는 일 — 도구를 불러오고, 데이터를 읽고, 시간순으로 나눈다
# 왜 하나 — 지난주와 같은 기준으로 채점해야 0.3918 과 비교할 수 있다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import pandas as pd                         # 표를 다루는 도구. pd 라고 부른다
import recsys                               # 이 수업용 도구 모음 — 읽기·분할·채점이 들어 있다

items, users, interactions = recsys.load()                  # 종목·투자자·거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 3주차부터 쓰던 시간순 분할

종목이름 = items.set_index("item_id")["name"].to_dict()        # to_dict = 딕셔너리로. {종목 번호: 이름}
이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # groupby = 사람별로 묶는다
전체_인기순위 = list(train["item_id"].value_counts().index)   # value_counts = 같은 값이 몇 번인지 센다

print(f"학습 {len(train):,}건 · 채점 {len(test):,}건 · 자른 날짜 {기준시점.date()}")

학습 7,095건 · 채점 991건 · 자른 날짜 2026-07-30


**결과 읽는 법**

- 거래 기록 8,086건을 2026-07-30 을 기준으로 둘로 나눈 결과입니다.
  - **학습 7,095건**은 자른 날짜 앞쪽 기록입니다. 모델이 배우는 데 씁니다.
  - **채점 991건**은 자른 날짜 뒤쪽 기록입니다. 추천이 맞았는지 확인하는 데 씁니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.1 준비 — 4주차 행렬분해를 다시 만든다</mark>

**6.1 에서 하는 일** — 4주차의 **사용자 요인(user factor, 285 × 8)** 과 **아이템 요인(item factor, 8 × 100)** 을 다시 만들고, 종목 100개 전부에 점수를 매기는 4주차 추천을 다시 돌립니다. 결과가 4주차의 **0.3918** 과 같은지 확인합니다.

**왜 4주차와 비교하나**
- 5주차 2단계 추천까지 섞으면, 점수가 바뀐 것이 상품 정보 때문인지 후보 40개 때문인지 가릴 수 없습니다.
- **한 번에 하나만 바꿔야** 비교가 정확합니다.

**변수 이름** — 4·5주차의 `사람표` · `종목표` 를 오늘은 공식 용어로 `사용자_요인` · `아이템_요인` 이라고 부르고, 「담았으면 1, 아니면 0」 표는 **사용자-아이템 행렬(user-item matrix)** 이므로 `상호작용_행렬` 이라고 부릅니다.

In [3]:
# 이 셀에서 하는 일 — 4주차와 똑같이 사용자-아이템 행렬을 만들고 잠재 요인 8개로 쪼갠다
# 왜 하나 — 오늘 쓰는 모델이 지난주 것과 같아야 상품 정보의 효과만 볼 수 있다

from sklearn.decomposition import TruncatedSVD      # 4주차에 쓴 행렬분해 도구

상호작용_행렬 = recsys.interaction_matrix(train, items)        # 사용자-아이템 행렬 — 담았으면 1, 아니면 0
모델 = TruncatedSVD(n_components=8, random_state=42)   # 잠재 요인 8개 — 지난주와 같은 설정
사용자_요인 = 모델.fit_transform(상호작용_행렬)                      # 사용자 요인(user factor) · 285 × 8
아이템_요인 = 모델.components_                            # 아이템 요인(item factor) · 8 × 100

사용자_목록 = list(상호작용_행렬.index)                         # 행렬의 행 이름(투자자 번호) 목록
아이템_목록 = list(상호작용_행렬.columns)                            # 행렬의 열 이름(종목 번호) 목록

print(f"사용자_요인 {사용자_요인.shape} · 아이템_요인 {아이템_요인.shape}   ← 지난주와 같은 크기")

사용자_요인 (285, 8) · 아이템_요인 (8, 100)   ← 지난주와 같은 크기


**결과 읽는 법**

- 4주차와 같은 크기의 요인 두 개가 만들어졌습니다.
  - `사용자_요인 (285, 8)` 은 투자자 285명 × 잠재 요인 8개입니다.
  - `아이템_요인 (8, 100)` 은 잠재 요인 8개 × 종목 100개입니다.

In [4]:
# 이 셀에서 하는 일 — 종목 100개의 점수를 받아, 점수가 큰 종목부터 늘어놓는 함수를 만든다
# 왜 하나 — 오늘 만드는 추천이 모두 「점수가 큰 종목부터 늘어놓기」를 한다

def 점수순으로_종목_정렬(점수):                       # def = 함수를 만든다. 종목 100개의 점수를 받는다
    """종목 100개의 점수를 받아, 점수가 큰 종목부터 종목 번호를 늘어놓은 목록을 돌려준다."""
    정렬된_종목 = []                                # 종목 번호를 점수 순서대로 담을 빈 목록
    for 자리 in np.argsort(-점수):                  # argsort(-점수) = 점수가 큰 것부터 그 자리 번호
        정렬된_종목.append(아이템_목록[자리])          # 자리 번호(0~99) → 종목 번호(I001~I100)
    return 정렬된_종목                              # 점수 1등 종목부터 100등 종목까지

In [5]:
# 이 셀에서 하는 일 — 4주차 추천 함수를 다시 만들고 U0001 에게 한 번 돌려 본다
# 왜 하나 — 오늘 모든 비교의 출발점이다

def 지난주_추천(사람, 이미):                          # def = 함수를 만든다
    """4주차 방식 — 종목 100개 전부에 점수를 매겨 높은 순으로 10개."""
    if 사람 not in 사용자_목록:                      # 사용자-아이템 행렬에 없는 신규 투자자는
        return recsys.take(전체_인기순위, 이미)      # 전체 인기 목록으로 준다
    점수 = 사용자_요인[사용자_목록.index(사람)] @ 아이템_요인   # @ = 행렬 곱. 사용자 요인 한 줄 × 아이템 요인
    return recsys.take(점수순으로_종목_정렬(점수), 이미)        # take = 이미 담은 것을 빼고 위에서 10개

print("U0001 에게 줄 추천 1등 :", 종목이름[지난주_추천("U0001", 이미_담은것["U0001"])[0]])

U0001 에게 줄 추천 1등 : TIGER 미국배당다우존스


**결과 읽는 법**

- U0001 이 이미 담은 종목을 빼고 남은 종목 가운데 1등입니다.
  - 점수가 가장 높은 종목은 U0001 이 이미 담은 종목이라 빠지고, 그다음 순위인 **TIGER 미국배당다우존스**가 1등이 되었습니다.

In [6]:
# 이 셀에서 하는 일 — 4주차 추천을 3주차와 같은 기준으로 채점한다
# 왜 하나 — 0.3918 이 나와야 지난주 모델을 제대로 다시 만든 것이다

지난주_점수, 인원 = recsys.recall_at_k(지난주_추천, train, test)   # 3주차부터 쓰던 채점 함수

print(f"4주차 추천 — Recall@10 = {지난주_점수:.4f}   (채점 대상 {인원}명)")

4주차 추천 — Recall@10 = 0.3918   (채점 대상 266명)


**결과 읽는 법**

- 6.1 에서 다시 만든 4주차 추천의 채점 결과입니다.
  - Recall@10 은 **0.3918** 로, 4주차에 나온 값과 같습니다.
  - 채점 대상은 **266명**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.2 4주차 추천의 위험도 — 안정형 U0001 의 10개</mark>

> 📖 **혼자 읽기** — 6.8 · 6.9 에서 같은 문제를 300명 전체 숫자로 다시 봅니다.

**6.2 에서 하는 일** — 6.1 의 4주차 모델이 안정형 투자자 U0001 에게 준 추천 10개에, 종목마다 **위험도**를 옆에 적어 봅니다.

**왜 확인하나**
- 4주차 모델은 **거래 기록만** 보고 추천했습니다. 종목의 위험도도, 투자자의 투자 성향도 모릅니다.
- 그래서 안정형 투자자에게 위험도가 높은 종목이 섞여 있을 수 있습니다. 실제로 그런지 봅니다.

**두 가지만 알면 됩니다**
- **위험도** — `items.csv` 의 `risk_level` 열. 종목마다 1 \~ 5 중 하나이고, **5 가 가장 위험**합니다
  - 예 — KODEX 단기채권 1 · KODEX 레버리지 5
- **투자 성향** — `users.csv` 의 `persona` 열. 안정형 · 성장형 · 공격형 중 하나입니다
  - 안정형은 원금 손실을 되도록 피하려는 투자자입니다. 고객이 계좌를 열 때 설문으로 정합니다

**출력에서 찾을 것** — 위험도가 4 이상이라 ⚠ 표시가 있는 줄이 몇 개인지 세어 보세요.

In [7]:
# 이 셀에서 하는 일 — U0001 의 추천 10개에 위험도를 붙여 본다
# 왜 하나 — 지난주 모델이 투자 성향을 전혀 보지 않는다는 것을 눈으로 확인한다

성향 = users.set_index("user_id")["persona"].to_dict()       # 딕셔너리 {투자자 번호: 안정형/성장형/공격형}
위험도 = items.set_index("item_id")["risk_level"].to_dict()   # 딕셔너리 {종목 번호: 1~5}
print(f"U0001 의 투자 성향 : {성향['U0001']}")

순위 = 1                                                      # 몇 등인지 세는 카운터
for 번호 in 지난주_추천("U0001", 이미_담은것["U0001"]):      # 지난주 모델이 준 10개
    표시 = "⚠" if 위험도[번호] >= 4 else " "                 # >= 크거나 같다. 4 이상이면 ⚠
    print(f"{순위:2d}. 위험도 {위험도[번호]} {표시} {종목이름[번호]}")   # :2d = 두 자리로 오른쪽 정렬
    순위 = 순위 + 1                                           # 다음 등수로

U0001 의 투자 성향 : 안정형
 1. 위험도 2   TIGER 미국배당다우존스
 2. 위험도 5 ⚠ TIGER 2차전지테마
 3. 위험도 3   Vanguard FTSE Developed Markets ETF
 4. 위험도 2   TIGER 은행고배당플러스TOP10
 5. 위험도 1   KODEX 단기채권
 6. 위험도 4 ⚠ 크래프톤
 7. 위험도 4 ⚠ Taiwan Semiconductor
 8. 위험도 2   Verizon Communications
 9. 위험도 2   Johnson & Johnson
10. 위험도 4 ⚠ Advanced Micro Devices


**결과 읽는 법**

- ⚠ 가 **4개**입니다. 안정형 U0001 에게 위험도 4 \~ 5 종목이 10개 중 4개 들어갔습니다.
  - 2등 TIGER 2차전지테마 — 위험도 5
  - 6등 크래프톤 · 7등 Taiwan Semiconductor · 10등 Advanced Micro Devices — 위험도 4

- 왜 이런 추천이 나왔나
  - 4주차 모델은 **협업 필터링**입니다. U0001 과 **거래 기록이 비슷한 다른 투자자들**이 담은 종목을 추천합니다.
  - 그 투자자 가운데 위험도가 높은 종목을 담은 사람이 있으면, 그 종목이 U0001 의 추천에도 들어옵니다.
  - 모델은 종목의 위험도도, U0001 의 투자 성향도 모릅니다. 거래 기록만 입력으로 받았기 때문입니다.

- 그래서 오늘 두 가지를 해 봅니다
  - 6.3 \~ 6.7 — 상품 정보를 **모델에 넣으면**(FM · MLP) 추천이 나아지는지 본다
  - 6.8 \~ 6.9 — 투자 성향에 맞지 않는 위험도의 종목을 추천 목록에서 **뺀다**

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.3 원핫 인코딩 — 상품 정보를 0/1 열로 펼치기</mark>

> 📖 **혼자 읽기** — 수업에서는 맨 위 「오늘 나오는 말」 2번의 표 한 장으로 설명합니다.

**6.3 에서 하는 일** — 종목마다 네 가지 상품 정보를 **0과 1로 된 열**로 바꿉니다. 오늘 쓰는 모델은 숫자만 입력으로 받는데, 상품 정보는 `레버리지` · `ETF` 같은 글자이기 때문입니다.

---

#### 자세히

**1. 예시 — 종목 세 개의 섹터만 펼쳐 보면**
- 변환 전과 변환 후 두 표는 맨 위 **「오늘 나오는 말」 2번**에 있습니다. 그 표를 보면서 읽어 주세요.
- 맨 윗줄 `레버리지 · 금융 · 소비재` 는 **열 이름**, 왼쪽 종목 이름은 **행 이름**입니다. 계산에 쓰는 값은 **가운데의 0 과 1 뿐**입니다.
- 한 줄에 1 이 **딱 하나만** 있습니다. 그래서 「원핫(one-hot)」이라고 부릅니다.

**2. 네 가지를 다 펼치면**
- 섹터 25 + 자산군 2 + 위험도 5 + 테마 9 = **41개 열**
- 열 이름 앞에 원래 열 이름이 붙습니다 — `sector_레버리지` · `theme_레버리지`. 「레버리지」가 섹터에도 테마에도 있어서, 붙이지 않으면 두 열의 이름이 같아집니다.
- 종목마다 정보가 네 가지이므로, 한 줄에 **1 이 정확히 4개**입니다(아래 출력의 I079).
- 위험도는 숫자이지만 **범주로 다룹니다.** 「위험도 4 인가 아닌가」만 봅니다.
- 이 변환은 pandas 의 `pd.get_dummies` 가 한 번에 합니다.

In [8]:
# 이 셀에서 하는 일 — 종목 100개의 상품 정보 네 가지를 0/1 열 41개로 펼친다
# 왜 하나 — 글자로는 계산을 못 한다. 0/1 이어야 평균도 곱셈도 된다

정보_열 = ["sector", "asset_class", "risk_level", "theme"]          # 섹터 · 자산군 · 위험도 · 테마
종목_정보 = items.set_index("item_id")[정보_열].astype(str)          # astype(str) = 위험도 숫자도 글자로 본다
아이템_피처 = pd.get_dummies(종목_정보).astype(int)                       # get_dummies = 원핫 인코딩. 결과가 아이템 피처 행렬(item feature matrix)
아이템_피처 = 아이템_피처.loc[아이템_목록]                                          # loc = 종목 순서를 사용자-아이템 행렬과 맞춘다
print(f"아이템_피처 {아이템_피처.shape}   ← 종목 100개 × 열 41개")

값이_1인_열 = []                                                         # 1 인 열 이름을 담을 빈 목록
for 열 in 아이템_피처.columns:                                            # 41개 열을 하나씩 본다
    if 아이템_피처.loc["I079", 열] == 1:                                   # == 같다. 1 인 열만 모은다
        값이_1인_열.append(열)                                            # append = 목록 끝에 붙인다
print(f"I079 {종목이름['I079']} → 1 인 열 : {값이_1인_열}")

아이템_피처 (100, 41)   ← 종목 100개 × 열 41개
I079 KODEX 레버리지 → 1 인 열 : ['sector_레버리지', 'asset_class_ETF', 'risk_level_5', 'theme_레버리지']


**결과 읽는 법**

- 첫 줄 `(100, 41)` 은 아이템 피처 행렬의 크기입니다.
  - 종목 100개 × 0/1 열 41개입니다.
- 둘째 줄은 I079 KODEX 레버리지 줄에서 값이 1 인 열 이름입니다.
  - 섹터 · 자산군 · 위험도 · 테마에서 하나씩, 모두 **4개**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.4 FM 의 생각 — 입력의 항목을 둘씩 짝지어 본다</mark>

> 🎯 **PPT 에서 설명합니다.** 이 절에는 실행할 셀이 없습니다.

**6.4 에서 하는 일** — FM 이 4주차 행렬분해와 무엇이 다른지 봅니다. 학습은 6.5 에서 합니다.

**1. 4주차 행렬분해는 두 가지만 입력으로 썼습니다**
- 투자자 번호와 종목 번호입니다.
- 그래서 「KODEX 레버리지의 위험도가 5 다」 같은 상품 정보를 넣을 자리가 없었습니다.

**2. FM 은 입력에 상품 정보를 더합니다**
- 「U0001 이 KODEX 레버리지를 담을까?」를 원핫 인코딩으로 쓰면, 한 줄에 1 이 **6개** 들어갑니다.

| 무엇 | 1 이 들어간 열 |
|---|---|
| 사용자 | `user_id_U0001` |
| 종목 | `item_id_I079` (KODEX 레버리지) |
| 섹터 | `sector_레버리지` |
| 자산군 | `asset_class_ETF` |
| 위험도 | `risk_level_5` |
| 테마 | `theme_레버리지` |

- 나머지 열 420개는 모두 0 입니다. 이 표는 6.5 에서 실제로 만들어 봅니다.

**3. FM 은 1 인 열을 둘씩 짝지어 점수를 계산합니다**
- 6개를 둘씩 짝지으면 6 × 5 ÷ 2 = **15짝**입니다.
  - 예 — (U0001, KODEX 레버리지) · (U0001, 위험도 5) · (KODEX 레버리지, 위험도 5) · …
- 짝마다 값을 계산해 모두 더한 것이 점수입니다. 계산은 `recsys.py` 가 하고, 오늘 직접 계산하지는 않습니다.

**4. 짝으로 보면 무엇이 좋은가**
- 위험도 5 는 공격형 투자자에게는 담을 이유가 되고, 안정형 투자자에게는 담지 않을 이유가 됩니다.
- 「위험도 5」 열 하나로는 이 차이를 나타낼 수 없습니다. 「U0001 × 위험도 5」 **짝**으로 볼 때 나타낼 수 있습니다.
- 사용자와 종목 두 열만 1 이면 짝은 (사용자, 종목) 하나뿐이고, 이때 FM 은 **4주차 행렬분해와 같은 계산**입니다. 그래서 FM 을 「행렬분해를 넓힌 모델」이라고 부릅니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.5 FM 학습 — 학습 데이터를 만들고, FM 을 학습해 채점한다</mark>

**6.5 에서 하는 일** — 세 단계로 FM 을 실제로 학습합니다. 학습은 「예측 → 정답과 비교 → 숫자를 조금 고친다」를 80번 반복하는 일입니다.

---

#### 자세히

| 단계 | 하는 일 | 쓰는 도구 |
|---|---|---|
| ① | 학습 데이터 만들기 — 담은 짝은 1, 담지 않은 짝은 0 | `recsys.training_rows` |
| ② | 학습 데이터를 원핫 인코딩 — 한 줄에 1 이 6개 | `recsys.one_hot` |
| ③ | FM 학습 → 285명 × 100종목에 점수 → 채점 | `recsys.FM` |

**1. 왜 「담지 않은 짝」이 필요한가 — 네거티브 샘플링**
- 모델은 **「이 짝이면 1, 저 짝이면 0」을 구별하는 법**을 배웁니다.
- 거래 기록에는 「담았다」만 있습니다. 1 만 있는 데이터로 배우면 모델은 **모든 짝에 1 이라고 답하면 다 맞는다**고 배웁니다. 아무것도 구별하지 못합니다.
- 그래서 사람마다 **담지 않은 종목을 무작위로 골라 0 으로 넣습니다.** 이것이 **네거티브 샘플링(negative sampling)** 입니다.
  - 「네거티브(negative)」 = 0 인 예시, 「샘플링(sampling)」 = 무작위로 고른다
  - 오늘은 사람마다 **담은 개수의 4배**를 0 으로 넣습니다.
- 「담지 않았다」가 「싫어한다」는 뜻은 아닙니다. 몰라서 안 담았을 수도 있습니다. 그래도 담은 종목보다는 덜 좋아할 가능성이 크다고 보고 0 으로 씁니다.


In [9]:
# 이 셀에서 하는 일 — ① 학습 데이터를 만든다. 담은 짝은 1, 담지 않은 짝은 0
# 왜 하나 — 0 이 있어야 모델이 「담을 짝」과 「안 담을 짝」을 구별하는 법을 배운다

학습_데이터 = recsys.training_rows(train, items, neg=4, seed=42)   # neg=4 = 담은 개수의 4배를 0 으로

print(f"학습 데이터 {len(학습_데이터):,}줄")
print(학습_데이터["label"].value_counts())       # label 열에서 1 과 0 이 각각 몇 줄인지 센다
학습_데이터.head(3)                               # 맨 위 세 줄

학습 데이터 27,750줄
label
0    20655
1     7095
Name: count, dtype: int64


,user_id,item_id,label
0,U0001,I006,1
1,U0001,I007,1
2,U0001,I013,1


**결과 읽는 법**

- 학습 데이터는 모두 **27,750줄**입니다. 한 줄이 「사용자 한 명 × 종목 하나」 짝입니다.
- `label` 은 정답 열입니다.
  - **1 이 7,095줄** — 학습 구간에서 실제로 담은 짝입니다.
  - **0 이 20,655줄** — 네거티브 샘플링으로 넣은 「담지 않은 짝」입니다.
  - 0 이 1 의 4배보다 조금 적습니다. 종목을 많이 담은 사람은 남은 종목이 적어서 4배를 다 채우지 못하기 때문입니다.
- 아래 표는 맨 위 세 줄입니다. 모두 U0001 이 담은 종목이라 `label` 이 1 입니다.

In [10]:
# 이 셀에서 하는 일 — ② 학습 데이터를 원핫 인코딩하고, 첫 줄에서 1 인 열을 본다
# 왜 하나 — FM 과 MLP 는 숫자만 받는다. 6.3 의 원핫 인코딩을 사용자 · 종목 번호에도 쓴다

X = recsys.one_hot(학습_데이터, items)   # 사용자 · 종목 · 상품 정보 네 가지를 0/1 열로 펼친다
y = 학습_데이터["label"]                 # 정답 — 1(담음) 또는 0(안 담음)

print(f"X {X.shape}   ← 학습 데이터 27,750줄 × 0/1 열")

첫줄_1인_열 = []                          # 첫 줄에서 1 인 열 이름을 담을 빈 목록
for 열 in X.columns:                      # 열을 하나씩 본다
    if X.iloc[0][열] == 1:                # iloc[0] = 맨 첫 줄. 1 인 열만 모은다
        첫줄_1인_열.append(열)
print(f"첫 줄 (U0001, I006 {종목이름['I006']}) → 1 인 열 {len(첫줄_1인_열)}개 :")
print(첫줄_1인_열)

X (27750, 426)   ← 학습 데이터 27,750줄 × 0/1 열
첫 줄 (U0001, I006 TIGER 리츠부동산인프라) → 1 인 열 6개 :
['user_id_U0001', 'item_id_I006', 'sector_리츠', 'asset_class_ETF', 'risk_level_2', 'theme_배당']


**결과 읽는 법**

- `X (27750, 426)` — 학습 데이터 27,750줄이 **0/1 열 426개**로 펼쳐졌습니다.
  - 사용자 285 + 종목 100 + 상품 정보 41 = **426**
  - 상품 정보 41개는 6.3 에서 만든 아이템 피처 행렬의 열과 같습니다.
- 첫 줄에서 1 인 열은 **6개**입니다 — 사용자 1 · 종목 1 · 섹터 1 · 자산군 1 · 위험도 1 · 테마 1.
  - 6.4 의 표와 같은 모양입니다. FM 은 이 6개를 둘씩 짝지어 15개 짝으로 점수를 계산합니다.
- `X` 는 입력, `y` 는 정답입니다. 머신러닝에서 입력을 `X`, 정답을 `y` 로 부르는 것이 관례입니다.

**③ FM 학습 — 아래 셀에서 일어나는 일**

1. **`recsys.FM(k=8, epochs=80)`** — FM 모델을 하나 만듭니다.
   - `k=8` — 열마다 정하는 잠재 요인의 개수. 4주차의 `n_components=8` 과 같은 뜻입니다.
   - `epochs=80` — **에폭(epoch)** 80. 학습 데이터 27,750줄을 처음부터 끝까지 **80번** 봅니다.
2. **`.fit(X, y)`** — 학습합니다.
   - 처음에는 잠재 요인이 아무렇게나 정해져 있습니다.
   - 한 줄씩 「예측한 확률」과 「정답(1 또는 0)」의 차이를 보고, 차이가 줄어드는 쪽으로 잠재 요인을 **조금씩** 고칩니다.
   - 이것을 80번 반복합니다. 한 번에 많이 고치면 숫자가 튀어서 조금씩 여러 번 고칩니다.
3. **`.predict(...)`** — 짝마다 **담을 확률(0 \~ 1)** 을 냅니다.
   - 285명 × 100종목 = **28,500개 짝** 모두에 확률을 매깁니다.
   - 확률이 높은 종목부터 10개를 추천합니다. 6.1 의 4주차 추천과 같은 방식입니다.
4. **두 번 학습합니다.**
   - 상품 정보를 **넣은** FM 과 **넣지 않은** FM(사용자 · 종목 열만) — 상품 정보가 도움이 됐는지 비교합니다.

- 무작위로 고르는 부분은 `seed=42` 로 고정해서, 누가 돌려도 같은 결과가 나옵니다.

**학습 시간** — 한 번에 10 \~ 15초쯤 걸립니다. 셀 왼쪽이 돌아가는 동안 기다려 주세요.

In [11]:
# 이 셀에서 하는 일 — 285 × 100 점수 행렬을 받아 채점하는 함수를 만든다
# 왜 하나 — 오늘 모델 네 개(행렬분해 · FM · MLP 1층 · MLP 2층)를 모두 같은 방식으로 채점한다

전체_짝 = recsys.all_pairs(사용자_목록, items)   # 점수를 매길 모든 짝 — 285명 × 100종목 = 28,500줄

def 점수_행렬로_채점(점수_행렬):
    """285 × 100 점수 행렬로 6.1 과 같은 방식의 추천을 만들고 Recall@10 을 잰다."""
    def 추천(사람, 이미):                                  # 함수 안에서 추천 함수를 하나 만든다
        if 사람 not in 사용자_목록:                         # 신규 투자자는 6.1 과 같이 인기 목록
            return recsys.take(전체_인기순위, 이미)
        점수 = 점수_행렬[사용자_목록.index(사람)]            # 이 사람의 종목 100개 점수
        return recsys.take(점수순으로_종목_정렬(점수), 이미)
    점수, _ = recsys.recall_at_k(추천, train, test)
    return 점수

print(f"행렬분해(4주차) 다시 채점 — Recall@10 = {점수_행렬로_채점(사용자_요인 @ 아이템_요인):.4f}")

행렬분해(4주차) 다시 채점 — Recall@10 = 0.3918


**결과 읽는 법**

- 6.1 의 4주차 추천을 새 함수로 다시 채점한 값입니다.
  - `사용자_요인 @ 아이템_요인` 은 285 × 100 점수 행렬입니다. 285명 각각의 종목 100개 점수입니다.
- **0.3918** 로 6.1 과 같습니다. 새 함수가 6.1 과 똑같이 채점한다는 확인입니다.
- 이제 FM 과 MLP 도 285 × 100 점수 행렬만 만들면 이 함수로 채점할 수 있습니다.

In [12]:
# 이 셀에서 하는 일 — ③ FM 을 두 번 학습해 채점한다. 상품 정보를 넣은 것과 넣지 않은 것
# 왜 하나 — FM 이 4주차보다 나은지, 상품 정보가 점수를 올렸는지 본다

import time                                         # 학습 시간을 재는 도구

시작 = time.time()
FM_모델 = recsys.FM(k=8, epochs=80).fit(X, y)                      # 상품 정보를 넣은 FM
FM_점수 = FM_모델.predict(recsys.one_hot(전체_짝, items))          # 28,500개 짝의 담을 확률
FM_점수 = FM_점수.reshape(len(사용자_목록), len(아이템_목록))       # reshape = 285 × 100 행렬로 다시 접는다
FM_결과 = 점수_행렬로_채점(FM_점수)
print(f"FM (상품 정보 넣음)   — Recall@10 = {FM_결과:.4f} · {time.time() - 시작:.0f}초")

시작 = time.time()
X_정보없음 = recsys.one_hot(학습_데이터, items, features=False)   # 사용자 · 종목 열만 — 상품 정보 없음
FM_모델_정보없음 = recsys.FM(k=8, epochs=80).fit(X_정보없음, y)
FM_점수_정보없음 = FM_모델_정보없음.predict(recsys.one_hot(전체_짝, items, features=False))
FM_점수_정보없음 = FM_점수_정보없음.reshape(len(사용자_목록), len(아이템_목록))
FM_결과_정보없음 = 점수_행렬로_채점(FM_점수_정보없음)
print(f"FM (상품 정보 없음)   — Recall@10 = {FM_결과_정보없음:.4f} · {time.time() - 시작:.0f}초")
print(f"행렬분해(4주차)       — Recall@10 = {지난주_점수:.4f}")

FM (상품 정보 넣음)   — Recall@10 = 0.3463 · 8초


FM (상품 정보 없음)   — Recall@10 = 0.3670 · 7초
행렬분해(4주차)       — Recall@10 = 0.3918


**결과 읽는 법**

- 세 줄 모두 같은 학습 구간으로 배우고, 같은 다음 달 기록으로 채점한 점수입니다.
- **FM (상품 정보 넣음)** — **0.3463** 입니다.
  - 4주차 행렬분해(0.3918)보다 **낮습니다.**
- **FM (상품 정보 없음)** — **0.3670** 입니다.
  - 상품 정보를 넣은 FM 보다 조금 높습니다. 사용자 · 종목 열만 쓰므로 4주차와 같은 재료를 썼습니다.
- **행렬분해(4주차)** — **0.3918** 로 셋 중 가장 높습니다.
- 끝의 초는 학습과 채점에 걸린 시간입니다. 4주차 TruncatedSVD 는 1초도 걸리지 않았습니다.
- ⚠ Colab 에서는 컴퓨터가 달라서 점수의 마지막 자리가 조금 다를 수 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.6 MLP — 층을 거쳐 확률을 내는 신경망, 층 1개와 2개</mark>

**6.6 에서 하는 일** — 같은 학습 데이터(`X`, `y`)로 **MLP** 를 학습합니다. 층이 1개인 것과 2개인 것을 비교합니다.

- **MLP** 는 계산 단계(**층**)를 여러 개 이어서 확률을 내는 모델입니다. 층이 많을수록 학습으로 정해야 하는 숫자가 늘어납니다.
- 층 1개는 `hidden_layer_sizes=(32,)`, 층 2개는 `(64, 32)` 입니다. 괄호 안 숫자는 층마다 만드는 값의 개수입니다.

---

#### 자세히

**FM 과 무엇이 다른가**

| | FM | MLP |
|---|---|---|
| 계산 모양 | **정해져 있음** — 짝마다 값을 계산해 더한다 | **정해져 있지 않음** — 층을 거치며 데이터에 맞는 계산을 찾는다 |
| 학습할 숫자 | 열 426개 × 잠재 요인 8개 ≈ **3,800개** | 1층 모델 ≈ **1만 4천 개** · 2층 모델 ≈ **2만 9천 개** |
| 장점 | 데이터가 적어도 비교적 안정적이다 | 데이터가 많으면 사람이 생각 못 한 관계까지 찾는다 |
| 점수가 왜 나왔는지 설명 | 짝별로 나눠 볼 수 있다 | 어렵다 — 숫자 수만 개가 섞여 있다 |

- 「학습할 숫자가 많다」 = 모델이 **더 복잡한 관계를 표현할 수 있다**는 뜻입니다. 그만큼 숫자를 정하는 데 **더 많은 데이터가 필요합니다.**
- 「왜 이 종목을 추천했나」를 고객이나 감독 기관이 물으면 답할 수 있어야 하므로, 금융에서는 설명하기 쉬운 모델이 유리합니다.


In [13]:
# 이 셀에서 하는 일 — MLP 를 층 1개, 층 2개로 학습해 각각 채점한다
# 왜 하나 — 층을 쌓아 모델을 깊게 만들면 점수가 오르는지 본다

from sklearn.neural_network import MLPClassifier    # scikit-learn 의 MLP
import warnings
warnings.filterwarnings("ignore")                   # 「30번으로는 학습이 덜 끝났다」 경고를 숨긴다. 결과에는 영향 없다

X_전체 = recsys.one_hot(전체_짝, items)               # 점수를 매길 28,500개 짝 — 학습 데이터와 같은 426열

시작 = time.time()
MLP_1층 = MLPClassifier(hidden_layer_sizes=(32,), max_iter=30, random_state=42).fit(X, y)
MLP_1층_점수 = MLP_1층.predict_proba(X_전체)[:, 1].reshape(len(사용자_목록), len(아이템_목록))
MLP_1층_결과 = 점수_행렬로_채점(MLP_1층_점수)
print(f"MLP 층 1개 (32)      — Recall@10 = {MLP_1층_결과:.4f} · {time.time() - 시작:.0f}초")

시작 = time.time()
MLP_2층 = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=30, random_state=42).fit(X, y)
MLP_2층_점수 = MLP_2층.predict_proba(X_전체)[:, 1].reshape(len(사용자_목록), len(아이템_목록))
MLP_2층_결과 = 점수_행렬로_채점(MLP_2층_점수)
print(f"MLP 층 2개 (64, 32)  — Recall@10 = {MLP_2층_결과:.4f} · {time.time() - 시작:.0f}초")

MLP 층 1개 (32)      — Recall@10 = 0.3283 · 3초


MLP 층 2개 (64, 32)  — Recall@10 = 0.2580 · 4초


**결과 읽는 법**

- **MLP 층 1개** — **0.3283** 입니다.
  - FM (상품 정보 넣음, 0.3463)과 비슷하고, 4주차 행렬분해(0.3918)보다 낮습니다.
- **MLP 층 2개** — **0.2580** 입니다.
  - 층을 하나 더 쌓았더니 점수가 **오히려 내려갔습니다.**
- 끝의 초는 학습과 채점에 걸린 시간입니다.
- 왜 깊게 만들었는데 내려갔는지는 6.7 에서 숫자를 하나 더 보고 생각해 봅니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.7 네 모델 비교 — 학습 데이터 정답률과 Recall@10 을 함께 본다</mark>

**6.7 에서 하는 일** — 오늘 만든 모델 네 개를 표 하나에 놓고, 숫자 두 개를 함께 봅니다. 학습 데이터 성적만 보고 모델을 고르면 MLP 2층을 고르게 되는데, 그 모델은 새 기록을 가장 못 맞힙니다.

---

#### 자세히

**1. 숫자 두 개**
- **학습 데이터 정답률** — 학습에 쓴 27,750줄에서 「담음(1) / 안 담음(0)」을 맞힌 비율
  - 확률이 0.5 보다 크면 「담음」, 아니면 「안 담음」으로 보고 정답과 비교합니다.
  - **이미 본 문제를 얼마나 잘 맞히나**입니다.
- **Recall@10** — 학습에 쓰지 않은 **다음 달 기록**을 얼마나 맞히나
  - **처음 보는 문제를 얼마나 잘 맞히나**입니다. 추천에서 진짜 중요한 숫자는 이쪽입니다.

**2. 참고할 기준**
- 학습 데이터의 74% 가 0 입니다(20,655 ÷ 27,750).
  - 그래서 모든 줄에 「안 담음」이라고만 답해도 정답률이 **0.74** 가 나옵니다.
  - 정답률은 0.74 보다 얼마나 높은지로 봅니다.
- 행렬분해(4주차)는 0/1 을 맞히는 모델이 아니어서 학습 데이터 정답률을 재지 않습니다.

In [14]:
# 이 셀에서 하는 일 — 네 모델의 학습 데이터 정답률과 Recall@10 을 표 하나로 모은다
# 왜 하나 — 「이미 본 문제」와 「처음 보는 문제」 성적을 나란히 비교한다

비교 = pd.DataFrame({
    "모델": ["행렬분해 (4주차)", "FM (상품 정보 넣음)", "MLP 층 1개", "MLP 층 2개"],
    "학습할 숫자": ["-", "약 3,800개", "약 1만 4천 개", "약 2만 9천 개"],
    "학습 데이터 정답률": ["-",
                          round(((FM_모델.predict(X) > 0.5) == y).mean(), 4),   # 0.5 보다 크면 1 로 보고 정답과 비교
                          round(MLP_1층.score(X, y), 4),                       # score = 정답률
                          round(MLP_2층.score(X, y), 4)],
    "Recall@10": [round(지난주_점수, 4), round(FM_결과, 4), round(MLP_1층_결과, 4), round(MLP_2층_결과, 4)],
})
비교

,모델,학습할 숫자,학습 데이터 정답률,Recall@10
0,행렬분해 (4주차),-,-,0.3918
1,FM (상품 정보 넣음),"약 3,800개",0.8451,0.3463
2,MLP 층 1개,약 1만 4천 개,0.8186,0.3283
3,MLP 층 2개,약 2만 9천 개,0.9785,0.2580


**결과 읽는 법**

- 한 줄이 모델 하나입니다. 위에서 아래로 갈수록 **학습할 숫자가 많은, 복잡한 모델**입니다.
- **학습 데이터 정답률** 열 — 이미 본 문제의 성적입니다.
  - FM 0.8451 · MLP 1층 0.8186 · **MLP 2층 0.9785**
  - MLP 2층은 학습 데이터 27,750줄 가운데 **98% 를 맞혔습니다.** 네 모델 중 가장 높습니다.
- **Recall@10** 열 — 처음 보는 다음 달 기록의 성적입니다.
  - **행렬분해 0.3918** 이 가장 높고, **MLP 2층 0.2580** 이 가장 낮습니다.
  - 1주차 인기 추천(0.2163)과 비교하면 MLP 2층은 인기 추천보다 조금 나은 정도입니다.
- 두 열의 순서가 **거꾸로**입니다. 이미 본 문제를 가장 잘 맞힌 모델이, 처음 보는 문제는 가장 못 맞혔습니다.

**🗣 수업에서 함께 보기**

MLP 2층은 학습 데이터를 **98%** 맞혔는데, 다음 달 기록은 네 모델 중 **가장 못 맞혔습니다**(Recall@10 0.2580).

모델 안에서 무슨 일이 일어난 걸까요? 오늘 나오는 말 가운데 어느 말로 설명할 수 있을까요?

- 힌트 — 학습 데이터에서 1 은 7,095줄이고, MLP 2층이 학습할 숫자는 약 2만 9천 개입니다. 두 숫자를 비교해 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.8 적합성 점검 — 성향에 맞지 않는 추천 세기</mark>

**6.8 에서 하는 일** — 300명에게 준 추천 3,000개 가운데 **투자 성향에 맞지 않는 것**이 몇 개인지 셉니다.

---

#### 자세히

**1. 적합성원칙**
- 금융소비자보호법 제17조 ③ — 「… 그 일반금융소비자에게 적합하지 아니하다고 인정되는 계약 체결을 권유해서는 아니 된다」.
- 추천 모델은 모든 고객에게 상품을 권하는 셈이므로 **금융 회사라면 이 기준을 신경 써야 합니다.**

**2. 오늘 쓰는 기준 (이 수업에서 정한 예시)**

| 투자 성향 | 추천해도 되는 위험도 |
|---|---|
| 안정형 | 3 까지 |
| 성장형 | 4 까지 |
| 공격형 | 5 까지 (제한 없음) |

- 실제 기준은 회사마다 다릅니다. 오늘은 이 표를 **지켜야 할 기준**으로 삼습니다.

**3. 어느 추천을 점검하나**
- 오늘 Recall@10 이 가장 높았던 **4주차 행렬분해 추천**(6.1 의 `지난주_추천`)을 점검합니다.
- FM · MLP 는 점수가 더 낮아서 오늘 최종 추천으로 쓰지 않습니다.

In [15]:
# 이 셀에서 하는 일 — 추천 3,000개 중 성향 기준을 넘는 것을 센다
# 왜 하나 — 6.2 에서 U0001 한 명으로 본 문제가 전체로는 얼마나 큰지 잰다

적합_기준 = {"안정형": 3, "성장형": 4, "공격형": 5}    # 딕셔너리 {성향: 추천해도 되는 최대 위험도}

def 안_맞는_개수(추천함수):                        # 추천 함수 자체를 받는다
    """300명의 추천 10개씩을 보고, 성향 기준을 넘는 종목 수를 센다."""
    개수 = 0                                       # 센 개수. 0 부터 시작한다
    for 사람 in users["user_id"]:                  # 300명을 한 명씩
        for 번호 in 추천함수(사람, 이미_담은것.get(사람, set())):   # get = 없으면 빈 집합
            if 위험도[번호] > 적합_기준[성향[사람]]:              # > = 크다. 기준보다 크면 센다
                개수 = 개수 + 1                    # 하나 더 센다
    return 개수                                    # 추천 3,000개 중 기준을 넘은 개수

print(f"4주차 추천 3,000개 중 성향에 맞지 않는 것 : {안_맞는_개수(지난주_추천)}개")

4주차 추천 3,000개 중 성향에 맞지 않는 것 : 205개


**결과 읽는 법**

- 300명에게 10개씩 준 4주차 추천 3,000개 가운데 적합 기준을 넘는 추천의 개수입니다.
  - **205개**가 투자 성향의 기준을 넘었습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 6.9 위험도 상한 — 후처리 필터링으로 추천을 거르고 상한 정하기</mark>

**6.9 에서 하는 일** — 4주차 추천 순서에서 **상한을 넘는 종목을 빼고**, 남은 종목 중 위에서부터 10개를 추천합니다.

---

#### 자세히

**1. 모델은 그대로입니다**
- 점수 계산은 4주차와 똑같습니다. **마지막에 걸러 내기만 합니다.**
- 빠진 자리는 **그다음 순위 종목**이 채웁니다. 그래서 추천은 언제나 10개입니다.

**2. 신규 15명도 똑같이 거릅니다**
- 신규 고객은 전체 인기 목록을 받는데, 인기 상위 10개에도 위험도 4 종목이 세 개 있습니다(Taiwan Semiconductor · Invesco QQQ Trust · KODEX 미국나스닥100).
  - 안정형 신규 고객에게는 기준(3 까지)을 넘는 종목입니다.
  - 투자 성향은 가입할 때 받으므로 신규 고객에게도 상한을 걸 수 있습니다.

**3. 상한과 기준을 따로 둔 이유**
- `적합_기준` — **지켜야 할 기준.** 6.8 에서 정했고 바꾸지 않습니다.
- `상한` — **추천 목록에 실제로 거는 값.** 이 절 뒤쪽 「상한 정하기」에서 여러 값으로 바꿔 봅니다.

In [16]:
# 이 셀에서 하는 일 — 4주차 추천 순서에서 상한을 넘는 종목을 빼고 10개를 준다
# 왜 하나 — 모델을 고치지 않고 마지막에 거르기만 해서 성향에 맞지 않는 추천을 없앤다

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}         # 추천 목록에 거는 상한 — 아래 「상한 정하기」에서 바꿔 본다

def 상한_추천(사람, 이미):                               # 4주차 추천 순서에서 상한을 넘는 종목만 빼는 함수
    """4주차 추천 순서에서 성향의 상한을 넘는 종목을 빼고 위에서 10개."""
    순서 = 전체_인기순위                                 # 신규는 인기 목록에서 시작한다
    if 사람 in 사용자_목록:                              # 기존 투자자는 4주차 점수 순서
        순서 = 점수순으로_종목_정렬(사용자_요인[사용자_목록.index(사람)] @ 아이템_요인)   # 6.1 과 같은 계산
    통과 = []                                            # 상한을 통과한 종목을 담을 빈 목록
    for 번호 in 순서:                                    # 점수가 높은 종목부터 하나씩
        if 위험도[번호] <= 상한[성향[사람]]:              # <= 작거나 같다. 상한 이하만 남긴다
            통과.append(번호)                            # 통과하면 목록에 붙인다
    return recsys.take(통과, 이미)                       # 통과한 것 중 이미 담은 것을 빼고 10개

In [17]:
# 이 셀에서 하는 일 — 상한을 건 추천을 채점하고, 성향에 맞지 않는 추천을 다시 센다
# 왜 하나 — 무엇을 얻었고(안 맞는 추천 수) 무엇을 내줬는지(점수) 함께 본다

상한_점수, _ = recsys.recall_at_k(상한_추천, train, test)

print(f"4주차 그대로   — Recall@10 = {지난주_점수:.4f} · 성향에 맞지 않는 추천 {안_맞는_개수(지난주_추천)}개")
print(f"위험도 상한    — Recall@10 = {상한_점수:.4f} · 성향에 맞지 않는 추천 {안_맞는_개수(상한_추천)}개")
print(f"→ 점수 {상한_점수 - 지난주_점수:+.4f}")          # :+.4f = 부호까지 붙여 소수 넷째 자리

4주차 그대로   — Recall@10 = 0.3918 · 성향에 맞지 않는 추천 205개
위험도 상한    — Recall@10 = 0.3899 · 성향에 맞지 않는 추천 0개
→ 점수 -0.0019


**결과 읽는 법**

- 첫 줄은 4주차 추천 그대로입니다.
  - 300명에게 준 추천 3,000개 중 **205개**가 투자 성향의 기준을 넘었습니다.
- 둘째 줄은 같은 추천에 위험도 상한을 건 결과입니다.
  - 기준을 넘는 추천이 **0개**가 되었습니다.
  - Recall@10은 0.3918 → **0.3899**입니다.
- 마지막 줄 `-0.0019`는 상한을 걸어서 **낮아진 점수**입니다.
  - 205개를 걸러 내는 대가로 Recall@10을 0.0019만큼 잃었다는 뜻입니다.

---

#### 📖 상한 정하기 — 점수와 성향에 맞지 않는 추천 수 비교

**여기서 하는 일** — 상한을 여러 값으로 바꿔 보고, **점수와 성향에 맞지 않는 추천 수**를 나란히 비교합니다.

**1. 무엇을 보고 고를까**
- 상한을 **느슨하게** 걸면 → 점수는 지키지만 기준을 넘는 추천이 남습니다.
- 상한을 **세게** 걸면 → 기준은 지키지만 고객이 실제로 담을 종목까지 빠져 점수가 내려갑니다.

**2. 답할 질문**
- 성향에 맞지 않는 추천을 **0개**로 만들면서 점수가 **가장 적게 내려가는** 상한은 무엇일까요?

In [18]:
# 이 셀에서 하는 일 — 상한을 바꿔 가며 점수와 안 맞는 추천 수를 함께 잰다
# 왜 하나 — 기준을 지키는 대가로 점수를 얼마나 내주는지 보고 상한을 고른다

실험 = [{"안정형": 5, "성장형": 5, "공격형": 5},       # 상한 없음 = 4주차
        {"안정형": 4, "성장형": 5, "공격형": 5},       # 안정형의 위험도 5 만 뺀다
        {"안정형": 3, "성장형": 4, "공격형": 5},       # 위의 기본값
        {"안정형": 2, "성장형": 3, "공격형": 4}]       # 기준보다 한 단계씩 더 세게

for 후보 in 실험:                                    # 상한 네 가지를 하나씩
    상한 = 후보                                        # 위 함수가 이 값을 보고 거른다
    점수, _ = recsys.recall_at_k(상한_추천, train, test)   # 위와 같은 채점
    print(f"안정 {후보['안정형']} · 성장 {후보['성장형']} · 공격 {후보['공격형']}   Recall@10 {점수:.4f} · 안 맞는 추천 {안_맞는_개수(상한_추천):3d}개")

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}         # 실험이 끝났으니 기본값으로 되돌린다

안정 5 · 성장 5 · 공격 5   Recall@10 0.3918 · 안 맞는 추천 205개
안정 4 · 성장 5 · 공격 5   Recall@10 0.3862 · 안 맞는 추천 154개
안정 3 · 성장 4 · 공격 5   Recall@10 0.3899 · 안 맞는 추천   0개
안정 2 · 성장 3 · 공격 4   Recall@10 0.2688 · 안 맞는 추천   0개


**결과 읽는 법**

1. **첫 줄** — 상한이 없습니다. 점수는 4주차와 같고, 성향에 맞지 않는 추천이 그대로 남습니다.

2. **둘째 줄** — 안정형에게서 위험도 5 만 뺐습니다. 아직 기준을 넘는 추천이 남습니다.
   - 점수가 셋째 줄보다 조금 낮은 것은 빠진 자리를 채운 종목이 정답이었는지에 따라 점수가 오르내리기 때문입니다. 0.004 정도는 작은 차이입니다.

3. **셋째 줄** — 상한을 기준과 같게 걸었습니다. **안 맞는 추천이 0개**가 되고, 점수는 0.3918 → 0.3899 로 **0.002 만** 내려갑니다.

4. **넷째 줄** — 기준보다 한 단계씩 더 세게 걸었습니다. 안 맞는 추천은 셋째 줄과 똑같이 0개인데 **점수만 더 내려갑니다.**

**한 줄로 정리하면** — 기준보다 세게 걸어도 더 지켜지는 것은 없고 점수만 깎입니다. **적합 기준과 같은 값을 상한으로 거는 것**이 점수가 가장 적게 내려가는 선택입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 6.10 오늘의 마무리</mark>

1. 상품 정보 네 가지를 **원핫 인코딩**으로 0/1 열 41개로 펼쳤고, 사용자 · 종목까지 펼쳐 한 줄에 1 이 6개인 입력을 만들었습니다.
2. **FM** 은 입력의 항목을 둘씩 짝지어 점수를 계산하는 모델입니다. 4주차 행렬분해에 상품 정보를 더해 넓힌 것입니다.
3. **MLP** 는 계산 단계(층)를 여러 개 이어서 확률을 내는 모델입니다. 층이 많을수록 깊다고 하고, 이것이 딥러닝입니다.
4. 우리 데이터에서는 **행렬분해 0.3918 > FM 0.3463 > MLP 1층 0.3283 > MLP 2층 0.2580** 이었습니다.
   - 복잡한 모델일수록 학습 데이터는 더 잘 맞혔지만, 다음 달 기록은 더 못 맞혔습니다 — **과적합**
5. 4주차 모델은 안정형 U0001 에게 **위험도 4\~5 종목을 네 개** 추천했습니다. 투자 성향별 **위험도 상한**을 걸어 성향에 맞지 않는 추천을 **205개에서 0개**로 줄였고, 그 대가로 점수가 **0.3918 → 0.3899** 로 조금 내려갔습니다.

**오늘 기억할 한 줄** — **작은 데이터에서는 단순한 모델이 낫습니다.** 상품 정보는 점수를 올리는 데가 아니라 **투자 성향에 맞지 않는 추천을 거르는 데** 썼습니다.

다음 주에는 지금까지 셀을 하나씩 눌러 한 일 — 모델 학습과 추천 — 을 **사람이 손대지 않아도 한 번에 돌아가도록** 묶고, 결과를 파일로 남깁니다.

In [19]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 점수가 내려간 주차도 그대로 적는다. 점수를 무엇과 바꿨는지 설명할 수 있으면 된다

recsys.record(6, "투자 성향별 위험도 상한", 상한_점수,
              note=f"4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 {안_맞는_개수(지난주_추천)}개 → 0개")

recsys.leaderboard(upto=6)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 6 · 투자 성향별 위험도 상한 · Recall@10 = 0.3899


,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 생성 + 순위 매기기,0.3382,2주차 세그먼트로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다
5,6,투자 성향별 위험도 상한,0.3899,4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 205개 → 0개


**결과 읽는 법**

- 첫 줄은 오늘 점수를 점수판에 남겼다는 확인입니다.
  - 레벨 6 · 투자 성향별 위험도 상한 · Recall@10 = **0.3899** 입니다.
- 아래 표는 1주차부터 6주차까지의 점수판입니다.
  - `level` 은 주차, `name` 은 그 주의 추천 방식, `recall_at_10` 은 점수, `note` 는 한 줄 설명입니다.
  - `note` 끝의 `...` 은 글이 길어서 화면에서 잘린 것입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 6.5 결과**

FM 에 상품 정보를 **넣은 것(0.3463)** 이 **넣지 않은 것(0.3670)** 보다 오히려 조금 낮았습니다.

FM 은 상품 정보를 넣을 수 있는 모델인데, 왜 상품 정보가 점수를 올리지 못했을까요?

- 힌트 — 「KODEX 레버리지의 위험도가 5 다」라는 정보는, **종목 번호(`item_id_I079`) 열만 보고도** 알 수 있는 정보일까요? 모델은 종목 번호 열의 잠재 요인을 거래 기록에서 이미 배웁니다.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 6.9 결과**

위험도 상한으로 성향에 맞지 않는 추천을 **205개 → 0개**로 줄였는데, Recall@10 은 0.3918 → 0.3899 로 **0.0019 만** 내려갔습니다.

추천 목록에서 종목이 많이 빠졌는데, 왜 점수는 거의 그대로일까요?

- 힌트 — 6.9 의 「빠진 자리는 그다음 순위 종목이 채웁니다」를 다시 읽어 보세요. 빠진 종목이 **원래 정답이었는지**가 열쇠입니다.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 3 · 6.9 결과**

회사가 기준을 바꿔 **「안정형은 위험도 2 까지」**로 더 엄격하게 정했다고 해 봅시다.

지금 상한(안정형 3 · 성장형 4 · 공격형 5)을 그대로 두면 어떻게 될까요? 상한을 어떻게 바꿔야 하고, 그 대가는 얼마일까요?

- 힌트 — 6.9 「상한 정하기」 표에서 **상한 숫자와 「안 맞는 추천」 개수**가 어떻게 함께 움직이는지 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북을 실제로 실행해서 얻은 값입니다.

**함께 보기 답 — 과적합입니다. 학습 데이터에만 지나치게 맞춰져서 처음 보는 기록을 못 맞혔습니다**

- MLP 2층이 학습할 숫자는 **약 2만 9천 개**입니다.
  - 학습 데이터에서 「담았다(1)」는 **7,095줄**뿐입니다. 정할 숫자가 배울 예시보다 4배쯤 많습니다.
- 숫자가 이렇게 많으면 모델은 「어떤 사람이 어떤 종목을 왜 담는가」라는 **규칙을 찾지 않아도** 됩니다.
  - 학습 데이터의 줄 하나하나에 **그대로 맞춰도** 정답을 맞힐 수 있기 때문입니다.
  - 그래서 학습 데이터 정답률은 **0.9785** 까지 올라갔습니다.
- 학습 데이터에 맞춰진 숫자는 다음 달 기록에 그대로 쓸 수 없습니다. 다음 달에 담을 종목은 학습 데이터에 없던 짝이기 때문입니다.
  - 그래서 Recall@10 은 **0.2580** 으로 가장 낮았습니다.
- 이것이 **과적합(overfitting)** 입니다. 4주차에서 잠재 요인을 64개로 늘렸을 때 점수가 **0.1778** 로 떨어진 것도 같은 이유였습니다.
- **한 줄로 정리하면** — 데이터가 작을 때 모델을 복잡하게 만들면 규칙 대신 데이터를 외웁니다. **작은 데이터에서는 단순한 모델이 낫습니다.**

**혼자 해 보기 1 답 — 상품 정보는 종목 번호만 알면 정해지는 정보라서, 모델이 거래 기록에서 이미 배운 것이었기 때문입니다**

- 종목 하나에 섹터 · 자산군 · 위험도 · 테마는 **하나씩 정해져 있습니다.**
  - `item_id_I079` 열이 1 이면 `risk_level_5` 열도 **언제나** 1 입니다.
  - 그래서 상품 정보 열은 종목 번호 열이 이미 가진 정보를 **다시 알려 줄 뿐**, 새 정보를 주지 않습니다.
- 종목 번호 열의 잠재 요인은 거래 기록에서 학습됩니다.
  - 「KODEX 레버리지를 담는 사람들이 어떤 사람들인가」를 배우면서, 위험도 5 라는 성격도 함께 배웁니다.
- 반대로 상품 정보 열 41개를 넣으면 **학습할 숫자가 늘어납니다**(열 385개 → 426개).
  - 새 정보 없이 숫자만 늘어서 점수가 조금 내려갔습니다. 6.7 의 「복잡할수록 낮아진다」와 같은 방향입니다.
  - 다만 차이(0.02)는 무작위로 고르는 종목이 달라지면 바뀔 정도로 작습니다. 확실한 것은 **「넣어도 오르지 않았다」**입니다.
- 상품 정보가 도움이 되는 경우는 따로 있습니다 — **거래 기록이 거의 없는 새 종목**입니다. 그 종목의 번호 열은 배울 기록이 없지만, 위험도 · 섹터 열은 다른 종목의 기록으로 이미 배워 두었습니다.
- **한 줄로 정리하면** — 거래 기록이 충분한 종목에는 상품 정보가 새로 알려 주는 것이 없습니다.

**혼자 해 보기 2 답 — 빠진 종목 대부분이 원래 정답이 아니었고, 빈자리를 채운 종목이 그만큼 다시 맞혔기 때문입니다**

- 채점 대상 266명의 추천에서 상한 때문에 빠진 종목은 **179개**입니다.
  - 그중 실제로 다음 달에 담은 정답은 **15개**뿐입니다.
- 빠진 자리에는 그다음 순위 종목 **179개**가 새로 들어왔습니다.
  - 그중 정답도 **15개**입니다. 잃은 만큼 다시 맞힌 셈입니다.
- 맞힌 개수가 같은데도 점수가 0.0019 내려간 것은 Recall@10 의 계산 방식 때문입니다.
  - Recall@10 은 사람마다 「맞힌 개수 ÷ 그 사람의 정답 개수」를 구한 뒤 평균을 냅니다.
  - 그래서 같은 한 개라도 사람마다 점수에 주는 무게가 다르고, 잃은 쪽의 무게가 조금 더 컸습니다.
- **한 줄로 정리하면** — 성향에 맞지 않는 추천은 대부분 어차피 맞히지 못하는 추천이었고, 그래서 걸러 내도 점수를 거의 잃지 않았습니다.

**혼자 해 보기 3 답 — 지금 상한으로는 324개가 새 기준을 넘고, 상한을 2 · 4 · 5 로 낮추면 0개가 되는 대신 점수가 0.3704 로 내려갑니다**

- 상한을 그대로(안정형 3) 두면, 안정형에게 간 위험도 3 종목이 모두 새 기준을 넘게 됩니다.
  - 실제로 세어 보면 성향에 맞지 않는 추천이 **324개**가 됩니다.
- 상한도 기준에 맞춰 **안정형 2 · 성장형 4 · 공격형 5** 로 바꿔야 합니다.
  - 그러면 안 맞는 추천은 다시 **0개**가 됩니다.
  - 대신 Recall@10 이 0.3899 → **0.3704** 로 0.0195 내려갑니다. 6.9 의 실험에서 본 것처럼, 상한이 엄격할수록 고객이 실제로 담을 종목도 함께 빠지기 때문입니다.
- **한 줄로 정리하면** — 기준이 바뀌면 상한도 같이 바꿔야 하고, 기준이 엄격할수록 점수가 더 많이 내려갑니다.